In [62]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import torch
np.random.seed(75)
torch.manual_seed(75)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name())
    torch.cuda.manual_seed(75)

NVIDIA GeForce RTX 4060 Laptop GPU


In [65]:
url = "https://raw.githubusercontent.com/aiplanethub/Datasets/master/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(url)


In [66]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
X = df.drop(["Churn" , "customerID"] , axis=1)
y = df["Churn"]

from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(X , y , test_size = 0.3 , stratify = y)



### Preprocessing

In [69]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Identify numerical and categorical columns
numerical_cols = X_train.select_dtypes(include=["number"]).columns
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns

# Numerical preprocessing
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both pipelines
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# Fit on training data and transform it
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data using the same fitted preprocessor
X_test_processed = preprocessor.transform(X_test)

print("X_train shape:", X_train_processed.shape)
print("X_test shape:", X_test_processed.shape)

X_train shape: (4930, 4668)
X_test shape: (2113, 4668)


In [70]:
# Convert X_train and X_test to NumPy arrays
X_train_np = X_train_processed.toarray()
X_test_np = X_test_processed.toarray()

# Convert target labels to NumPy arrays
y_train_np = y_train.map({"No": 0, "Yes": 1}).to_numpy(dtype=np.float32)
y_test_np = y_test.map({"No": 0, "Yes": 1}).to_numpy(dtype=np.float32)

# Check shapes
print("X_train:", X_train_np.shape)
print("X_test:", X_test_np.shape)
print("y_train:", y_train_np.shape)
print("y_test:", y_test_np.shape)

# Check data types
print(type(X_train_np))
print(type(y_train_np))

X_train: (4930, 4668)
X_test: (2113, 4668)
y_train: (4930,)
y_test: (2113,)
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [71]:
X_train_tensor = torch.from_numpy(X_train_np).float()
X_test_tensor = torch.from_numpy(X_test_np).float()
y_train_tensor = torch.from_numpy(y_train_np).float()
y_test_tensor = torch.from_numpy(y_test_np).float()

In [73]:
X_train_tensor.shape , X_test_tensor.shape , y_train_tensor.shape , y_test_tensor.shape

(torch.Size([4930, 4668]),
 torch.Size([2113, 4668]),
 torch.Size([4930]),
 torch.Size([2113]))

In [74]:
BATCH_SIZE = 32
NUM_WORKER = 3

from torch.utils.data import TensorDataset , DataLoader

train_data = TensorDataset(X_train_tensor , y_train_tensor)
test_data = TensorDataset(X_test_tensor , y_test_tensor)


train_loader = DataLoader(
    dataset = train_data,
    batch_size = BATCH_SIZE,
    num_workers = NUM_WORKER,
    shuffle = True
)

test_loader = DataLoader(
    dataset = test_data,
    batch_size = BATCH_SIZE,
    num_workers = NUM_WORKER,
    shuffle = True
)

In [75]:
import torch.nn as nn

class ChurnPredictor(nn.Module):
    def __init__(self , input_shape : int):
        super().__init__()

        self.layer1 = nn.Linear(input_shape , 32)
        self.layer2 = nn.Linear(32 , 16)
        self.layer3 = nn.Linear(16 , 8)
        self.layer4 = nn.Linear(8 , 1)

        self.relu = nn.ReLU()

    def forward(self , X):

        # input : Layer 1
        x = self.layer1(X)
        x = self.relu(x)

        # hidden : Layer 2
        x = self.layer2(x)
        x = self.relu(x)

        # hidden : Layer 3
        x = self.layer3(x)
        x = self.relu(x)

        # output : Layer 4
        x = self.layer4(x)
        x = self.relu(x)

        return x.squeeze(-1)



In [76]:
import torch.optim as optim

model = ChurnPredictor(X_train_tensor.shape[1])

optimizer = optim.SGD(
    lr = 0.001,
    params = model.parameters()
)

### Training Loop

In [83]:
EPOCHS = 10

loss_fn = nn.BCEWithLogitsLoss()

loss_per_epoch = []

for epoch in range(1 , EPOCHS+1):

    model.train()
    total_loss = 0

    for X_train_batch , y_train_batch in train_loader:

        y_pred = model(X_train_batch)

        batch_loss = loss_fn(y_pred , y_train_batch)
        
        optimizer.zero_grad()

        batch_loss.backward()

        optimizer.step()

        total_loss+=batch_loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch : {epoch}/{EPOCHS} | Training Loss : {avg_loss:.3f}")



    model.eval()
    total_loss = 0
    with torch.inference_mode():
        for X_test_batch , y_test_batch in test_loader:
            batch_loss = model(X_test_batch)
            loss = loss_fn(batch_loss , y_test_batch)
            
            total_loss += loss.item()
        avg_loss = total_loss / len(test_loader)

        print(f"Epoch : {epoch}/{EPOCHS} | Validation Loss : {avg_loss:.3f}")

    print()

Epoch : 1/10 | Training Loss : 0.693
Epoch : 1/10 | Validation Loss : 0.693

Epoch : 2/10 | Training Loss : 0.693
Epoch : 2/10 | Validation Loss : 0.693

Epoch : 3/10 | Training Loss : 0.693
Epoch : 3/10 | Validation Loss : 0.693

Epoch : 4/10 | Training Loss : 0.693
Epoch : 4/10 | Validation Loss : 0.693

Epoch : 5/10 | Training Loss : 0.693
Epoch : 5/10 | Validation Loss : 0.693

Epoch : 6/10 | Training Loss : 0.693
Epoch : 6/10 | Validation Loss : 0.693

Epoch : 7/10 | Training Loss : 0.693
Epoch : 7/10 | Validation Loss : 0.693

Epoch : 8/10 | Training Loss : 0.693
Epoch : 8/10 | Validation Loss : 0.693

Epoch : 9/10 | Training Loss : 0.693
Epoch : 9/10 | Validation Loss : 0.693

Epoch : 10/10 | Training Loss : 0.693
Epoch : 10/10 | Validation Loss : 0.693

